# Part 1: Python & Data Foundations
Upload `appointments.csv`, `clinics.csv`, `followup_notes.csv` into a folder called `data/` in Colab (left Files panel -> new folder `data` -> upload). Do not edit the originals.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
print(os.listdir("data"))

## Task 1.1: csv module only

In [ ]:
import csv, json

with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    rows = list(reader)
print("Row count:", len(rows))

def parse_row(row):
    r = dict(row)
    r["age"] = int(r["age"])
    r["fee_inr"] = int(r["fee_inr"])
    r["prior_no_shows"] = int(r["prior_no_shows"])
    r["distance_km"] = float(r["distance_km"])
    return r

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("accepted:", len(accepted), "rejected:", len(rejected))   # expect 988 / 32
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

## Task 1.2: pandas, pass 1 (starts again from the raw file)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/appointments.csv")
print(df.shape)                      # (1020, 14)
df.info()
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())          # 20
print("raw channel spellings:", df["booking_channel"].nunique())   # 16

print("start:", len(df))
df = df.drop_duplicates()
print("after (a) drop duplicates:", len(df))          # 1000

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("after (b) channel clean:", len(df), "| channels:", sorted(df["booking_channel"].dropna().unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("after (c) dates converted:", len(df))

df = df.dropna(subset=["appt_date", "status"])
print("after (d) drop missing appt_date/status:", len(df))   # 990

## Task 1.3: pass 2 (validity, gaps, outliers)

In [ ]:
# (a) impossible ages -> missing
bad_age = (df["age"] < 1) | (df["age"] > 100)
print("invalid ages:", bad_age.sum())                 # 5
df.loc[bad_age, "age"] = np.nan
print("missing ages now:", df["age"].isna().sum())    # 17

# (b) fill with medians
age_med = df["age"].median()
dist_med = df["distance_km"].median()
print("median age:", age_med, "| median distance:", dist_med)   # 48 / 8.25
df["age"] = df["age"].fillna(age_med).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_med)

# (c) IQR fences + np.clip
for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    clipped = np.clip(df[col], lo, hi)
    n_capped = int((clipped != df[col]).sum())
    print(f"{col}: Q1={q1}, Q3={q3}, lower={lo}, upper={hi}, capped={n_capped}")
    df[col] = clipped

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("max fee:", df["fee_inr"].max())               # 1219

## Task 1.4: features + NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())     # 1 to 45
print("no-shows:", df["no_show"].sum(), "of", len(df), f"({round(100*df['no_show'].mean(),1)}%)")  # 218, 22.0%

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("(a) column means:", np.round(X.mean(axis=0), 2))       # ~[48.17 9.50 635.71 11.82]

mask = X[:, 3] > 14
print("(b) lead_days > 14:", mask.sum(), "rows,", round(100 * mask.mean(), 1), "%")   # 296, 29.9

x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("(c) |z| > 2 for distance_km:", (np.abs(z) > 2).sum())  # 53

## Task 1.5: enrich clinics from Open-Meteo

In [ ]:
import requests

URL = "https://api.open-meteo.com/v1/forecast"
clinics = pd.read_csv("data/clinics.csv")
name_col = clinics.columns[1]       # clinic name column
temps = []

for _, c in clinics.iterrows():
    temp, reason = None, ""
    for attempt in range(2):        # first try + one retry
        try:
            r = requests.get(URL, params={"latitude": c["latitude"], "longitude": c["longitude"],
                                          "hourly": "temperature_2m", "forecast_days": 1}, timeout=10)
            if r.status_code != 200:
                reason = f"status {r.status_code}"
                continue
            js = r.json()
            if "hourly" not in js:
                reason = "no 'hourly' in reply"
                continue
            temp = js["hourly"]["temperature_2m"][0]
            if temp is None:
                reason = "first temperature value is null"
                continue
            break
        except requests.exceptions.RequestException as e:
            reason = str(e)
    if temp is None:
        print(f"FAILED {c[name_col]}: {reason}")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

## Task 1.6: export clean table + SQL load script

In [ ]:
cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
        "prior_no_shows", "status", "lead_days", "no_show"]

out = df.sort_values("appointment_id").reset_index(drop=True).copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[cols]

print("appointment_id unique:", out["appointment_id"].is_unique, "| distinct:", out["appointment_id"].nunique())
print("missing values per column:\n", out.isnull().sum())
assert out["appointment_id"].is_unique and out.isnull().sum().sum() == 0

out.to_csv("appointments_clean.csv", index=False)
print(out.shape)       # (990, 16)

clinics_e = pd.read_csv("clinics_enriched.csv")
lines = []
for r in clinics_e.itertuples(index=False):
    cid, cname, city, tier, lat, lon, t = r[0], r[1], r[2], r[3], r[4], r[5], r[6]
    t_sql = "NULL" if pd.isna(t) else t
    lines.append(f"INSERT INTO clinics VALUES ('{cid}','{cname}','{city}','{tier}',{lat},{lon},{t_sql});")

for r in out.itertuples(index=False):
    lines.append(
        f"INSERT INTO appointments VALUES ('{r.appointment_id}','{r.patient_id}','{r.clinic_id}',"
        f"'{r.booked_on}','{r.appt_date}',{r.age},'{r.gender}',{r.distance_km},{r.fee_inr},"
        f"'{r.booking_channel}','{r.visit_type}','{r.reminder_sent}',{r.prior_no_shows},"
        f"'{r.status}',{r.lead_days},{r.no_show});")

with open("load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")
print("INSERT statements:", len(lines))     # 998

## Task 2.2 (Python check): groupby counts must match the SQL

In [ ]:
chk = (out.groupby("clinic_id")
          .agg(appointments=("appointment_id", "count"), no_shows=("no_show", "sum"))
          .reset_index()
          .merge(clinics_e.iloc[:, [0, 1, 2]], left_on="clinic_id", right_on=clinics_e.columns[0]))
chk["no_show_pct"] = (100 * chk["no_shows"] / chk["appointments"]).round(1)
print(chk.sort_values("no_show_pct", ascending=False))